# 🛡️ Theft & Shoplifting Detection — Deep Learning

**Module Deep Learning | Encadrant : M. Abdallah Khemais**

---

### 🎯 Objectif
Classifier automatiquement des images de vidéosurveillance :
- 🔴 `shoplifting` — vol détecté
- 🟢 `normal` — comportement normal

### 📦 Dataset (Kaggle)
> **Comment ajouter votre dataset :**
> 1. Allez sur [kaggle.com/datasets](https://www.kaggle.com/datasets) → **New Dataset**
> 2. Uploadez votre dossier `data/` (avec `train/valid/test`)
> 3. Dans ce notebook : cliquez **Add Data** → sélectionnez votre dataset
> 4. Le chemin sera `/kaggle/input/<nom-du-dataset>/`

### 🗺️ Plan
| # | Section |
|---|---------|
| 1 | Setup & Configuration |
| 2 | Chargement Dataset |
| 3 | Exploration |
| 4 | Prétraitement & Augmentation |
| 5 | Fonctions utilitaires |
| 6 | Baseline CNN |
| 7 | ResNet50 |
| 8 | EfficientNet-B0 |
| 9 | Fine-Tuning Progressif |
| 10 | Combinaison Gagnante |
| 11 | Comparaison Finale |
| 12 | Évaluation & Sauvegarde |

---
## 1. ⚙️ Setup

In [ ]:
import os, time, copy, gc, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, f1_score, precision_recall_curve
)
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

NUM_EPOCHS  = 10
BATCH_SIZE  = 32
IMG_SIZE    = 224
NUM_CLASSES = 2

print('=' * 50)
print(f'  Device : {device}')
if torch.cuda.is_available():
    print(f'  GPU    : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'  Batch  : {BATCH_SIZE} | Epochs : {NUM_EPOCHS}')
print('=' * 50)
print('✅ Setup terminé !')

---
## 2. 📂 Chargement du Dataset

In [ ]:
# ── Chemin exact du dataset Kaggle ───────────────────────────
DATA_DIR  = '//kaggle/input/datasets/raniadabbek/projet/deep_learning'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR   = os.path.join(DATA_DIR, 'valid')
TEST_DIR  = os.path.join(DATA_DIR, 'test')

# ── Vérification ──────────────────────────────────────────────
print('📂 Vérification des chemins :')
for name, path in [('DATA ', DATA_DIR), ('TRAIN', TRAIN_DIR), ('VAL  ', VAL_DIR), ('TEST ', TEST_DIR)]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'  {status} {name} : {path}')

assert os.path.exists(TRAIN_DIR), f'❌ Dossier train introuvable : {TRAIN_DIR}'

# ── Comptage images ───────────────────────────────────────────
print('\n📊 Contenu du dataset :')
for split, path in [('train', TRAIN_DIR), ('valid', VAL_DIR), ('test', TEST_DIR)]:
    if path and os.path.exists(path):
        classes = sorted([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))])
        total   = 0
        print(f'\n  [{split}]')
        for cls in classes:
            n = len([f for f in os.listdir(os.path.join(path, cls))
                     if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp'))])
            print(f'    {cls:20s} : {n:6,} images')
            total += n
        print(f'    {"TOTAL":20s} : {total:6,} images')

CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
print(f'\n✅ Classes détectées : {CLASS_NAMES}')

---
## 3. 🔍 Exploration des Données

In [ ]:
import os, shutil, time
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets

cell_start = time.time()

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────
BASE_DIR    = Path('/kaggle/input/datasets/raniadabbek/projet/deep_learning')
OUTPUT_DIR  = Path('/kaggle/working/deep_learning_clf')   # ✅ dossier accessible en écriture
CLASS_NAMES = ['normal', 'theft']

# ──────────────────────────────────────────────
# 🔄  Conversion YOLO → ImageFolder
# ──────────────────────────────────────────────
def yolo_to_imagefolder(base_dir, output_dir, class_names):
    if output_dir.exists():
        shutil.rmtree(output_dir)

    converted = {'train': 0, 'valid': 0, 'test': 0}
    skipped   = 0

    for split in ['train', 'valid', 'test']:
        img_dir = base_dir / split / 'images'
        lbl_dir = base_dir / split / 'labels'
        if not img_dir.exists():
            print(f"  ⚠️  {split}/images introuvable, ignoré.")
            continue

        for img_path in sorted(img_dir.iterdir()):
            if img_path.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp','.webp'}:
                continue

            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                skipped += 1
                continue

            with open(lbl_path) as f:
                first = f.readline().strip()
            if not first:
                skipped += 1
                continue

            class_id   = int(first.split()[0])
            class_name = class_names[class_id] if class_id < len(class_names) else f'class_{class_id}'

            dest = output_dir / split / class_name
            dest.mkdir(parents=True, exist_ok=True)
            shutil.copy2(img_path, dest / img_path.name)
            converted[split] += 1

    return converted, skipped

print("🔄 Conversion YOLO → ImageFolder …")
converted, skipped = yolo_to_imagefolder(BASE_DIR, OUTPUT_DIR, CLASS_NAMES)

for split, n in converted.items():
    print(f"  {split:6s} → {n} images copiées")
print(f"  ⚠️  {skipped} images ignorées (label manquant)")

TRAIN_DIR = OUTPUT_DIR / 'train'
VALID_DIR = OUTPUT_DIR / 'valid'
TEST_DIR  = OUTPUT_DIR / 'test'

# ──────────────────────────────────────────────
# 📊  Distribution + visualisation
# ──────────────────────────────────────────────
def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.numpy().transpose((1, 2, 0))
    return np.clip(std * img + mean, 0, 1)

preview_ds = datasets.ImageFolder(TRAIN_DIR)
counts     = Counter(preview_ds.targets)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('📊 Distribution des Classes — Train', fontsize=14, fontweight='bold')

colors = ['#2ECC71', '#E74C3C']
bars   = axes[0].bar(CLASS_NAMES,
                     [counts[i] for i in range(len(CLASS_NAMES))],
                     color=colors, edgecolor='black', linewidth=0.8)
for bar, v in zip(bars, [counts[i] for i in range(len(CLASS_NAMES))]):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 20, str(v),
                 ha='center', fontweight='bold', fontsize=12)
axes[0].set_ylabel("Nombre d'images")

axes[1].pie([counts[i] for i in range(len(CLASS_NAMES))],
            labels=CLASS_NAMES, colors=colors,
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('Proportion')

plt.tight_layout()
plt.show()

ratio = max(counts.values()) / min(counts.values())
print(f'⚠️  Ratio déséquilibre : {ratio:.2f}x  '
      f'{"→ class weights activés" if ratio > 1.5 else "→ dataset équilibré"}')

# ──────────────────────────────────────────────
# ⏱️  Temps d'exécution
# ──────────────────────────────────────────────
elapsed = time.time() - cell_start
mins, secs = divmod(elapsed, 60)
print(f"\n⏱️  Temps d'exécution : {int(mins)}m {secs:.2f}s")

In [ ]:
# ── Exemples d'images ─────────────────────────────────────────
basic_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
preview_full = datasets.ImageFolder(TRAIN_DIR, transform=basic_tf)

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('🖼️ Exemples d\'images', fontsize=14, fontweight='bold')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    indices = [i for i, (_, l) in enumerate(preview_full.samples) if l == cls_idx][:6]
    for col, idx in enumerate(indices):
        img, _ = preview_full[idx]
        axes[cls_idx][col].imshow(denormalize(img))
        axes[cls_idx][col].set_title(cls_name, fontsize=9,
            color='green' if cls_name == 'normal' else 'red')
        axes[cls_idx][col].axis('off')
plt.tight_layout(); plt.show()